# §2.5 robust validation: held-out test + measurement-gap baseline

**Why this notebook exists.** The first §2.5 notebook (`08_sec2.5_data_driven_generation.ipynb`) calibrated the C++ generator on the same 324 real plants it then KS-compared against. That's data peeking. It also reports KS p-values at N=324, where any 2° mean shift gives p<0.001 — not informative. This notebook does three things differently:

1. **80/20 train/test split.** The sorghum config XML is derived from the train half only; the synthetic plants are then KS-compared against the **held-out test half**. Single-shot validation, no peeking.
2. **Measurement-gap baseline.** We also compute KS / W₁ between the test plants' gold-standard θ and the **PhenoFrame trait extractor's** θ on procedural fits of the SAME plants. That's the floor: even with no generation at all, just running the procedural-trait pipeline on the real skeleton produces some disagreement with the voxel-PCA gold standard. The right question is whether the generator's residual is mostly *that* gap, or mostly generator-vs-population mismatch.
3. **Wasserstein-1 distance** (in degrees, interpretable units) as the primary metric. KS statistic as secondary. KS p-values dropped, since at this N they aren't informative.

**What this notebook does NOT change** vs `08_*`: the C++ generator binary itself, the 14-parameter Gaussian + 4-latent loadings architecture, and the canopy-curve representation (base + bottom/middle/top offset).

In [ ]:
from __future__ import annotations

import sys, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

import os
REPO_ROOT = Path(os.environ.get('PHENOFRAME_EXPERIMENT_ROOT', str(Path.cwd().parents[1] if Path.cwd().name == 'results' else Path.cwd())))
sys.path.insert(0, str(REPO_ROOT))

from experiments import data_loaders as dl
from experiments.paths import assert_data_present
from phenoframe import compute_traits_from_descriptor, MaizeGenerator

assert_data_present()
OUT = Path(os.environ.get('PHENOFRAME_OUTPUT_DIR', str(REPO_ROOT / 'experiments' / 'results' / 'outputs')))
FIG = Path(os.environ.get('PHENOFRAME_FIGURE_DIR', str(REPO_ROOT / 'experiments' / 'results' / 'figures')))
FIG.mkdir(exist_ok=True, parents=True)

# Locate the C++ generator binary via the wrapper's resolution order
# (PHENOFRAME_GENERATOR_EXE env var -> conventional paths -> system PATH).
GENERATOR = MaizeGenerator()
PAPER = REPO_ROOT.parent / 'paper'
BASE_CONFIG = PAPER / 'configs' / 'maize_generator_config.xml'
assert BASE_CONFIG.exists(), f'base config not found: {BASE_CONFIG}'

RNG_SEED = 20260601
RNG = np.random.default_rng(RNG_SEED)
ARCH_PARAMS = ['leaf_length', 'leaf_angle', 'droopiness', 'distance']
LOWER = list(range(5))
print('Imports OK.')
print(f'Generator: {GENERATOR}')


## 1. 80/20 train/test split of the real plant population

Split is over **plant IDs**, not leaves — so no leaf from a train plant ever appears in the test set. Seed is fixed for reproducibility.

In [ ]:
# Use the same filtered E5 population as 08_sec2.5
raw = pd.read_csv(OUT / 'e5_fitted_params.csv')
rms_cap = raw['rms_error_cm'].quantile(0.90)
real = raw[(raw['leaf_voxel_count'] >= 20) & (raw['rms_error_cm'] <= rms_cap)].copy()
real = real[['plant_id', 'leaf_index'] + ARCH_PARAMS].copy()
all_plants = real['plant_id'].drop_duplicates().sort_values().reset_index(drop=True)
n_total = len(all_plants)

# Deterministic 80/20 split
perm = RNG.permutation(n_total)
n_train = int(round(n_total * 0.80))
train_idx = perm[:n_train]
test_idx  = perm[n_train:]
train_plants = set(all_plants.iloc[train_idx])
test_plants  = set(all_plants.iloc[test_idx])
print(f'Total plants: {n_total}    train: {len(train_plants)}    test: {len(test_plants)}')

real_train = real[real['plant_id'].isin(train_plants)].copy()
real_test  = real[real['plant_id'].isin(test_plants)].copy()
print(f'train leaves: {len(real_train):,}    test leaves: {len(real_test):,}')

MAX_NODE = int(real['leaf_index'].max())

## 2. Derive C++ generator config from TRAIN ONLY

Replicates the logic of `derive_sorghum_config.py` (v3 calibration: gold-standard θ for leafAngle, uniform-noise correction for tillerAzimuthNoise, leafJitterScale = 0.60), but the only inputs are train-set plants. The test plants are never touched here.

In [ ]:
# Per-plant statistics (train only)
per_plant_train = real_train.groupby('plant_id').agg(
    n_leaves=('leaf_index', 'count'),
    height=('distance', 'sum'),
)
n_leaves_mean = float(per_plant_train['n_leaves'].mean())
n_leaves_std  = float(per_plant_train['n_leaves'].std())
height_mean   = float(per_plant_train['height'].mean())
height_std    = float(per_plant_train['height'].std())
print(f'Train: n_leaves = {n_leaves_mean:.2f} +/- {n_leaves_std:.2f}    '
      f'height = {height_mean:.3f} +/- {height_std:.3f} m')

# Per-node canopy curves (train only)
def fit_curve(param, multiplicative=False, n_bins=11):
    df = real_train.merge(per_plant_train['n_leaves'].rename('plant_n_leaves'),
                          left_on='plant_id', right_index=True)
    df['t'] = df['leaf_index'] / np.maximum(df['plant_n_leaves'] - 1, 1)
    bins = np.linspace(0, 1, n_bins + 1)
    df['_bin'] = pd.cut(df['t'], bins, include_lowest=True, labels=False)
    binned = df.groupby('_bin')[param].mean().reindex(range(n_bins))
    base = float(binned.iloc[n_bins // 2])
    std  = float(real_train[param].std())
    bot, mid, top = float(binned.iloc[0]), base, float(binned.iloc[-1])
    if multiplicative:
        return base, std, bot / base, mid / base, top / base
    return base, std, bot - mid, 0.0, top - mid

ll_base, ll_std, ll_bot, ll_mid, ll_top = fit_curve('leaf_length', multiplicative=True)
dr_base, dr_std, dr_bot, _,      dr_top = fit_curve('droopiness',  multiplicative=False)

# leafAngle from gold-standard theta (train plants only)
e1b = pd.read_csv(OUT / 'e1b_per_leaf_pymaize_vs_gold.csv')
gold_train = e1b[e1b['plant_id'].isin(train_plants)][['plant_id','leaf_index','theta_gold']].dropna().copy()
gold_train = gold_train.merge(per_plant_train['n_leaves'].rename('plant_n_leaves'),
                              left_on='plant_id', right_index=True)
gold_train['t'] = gold_train['leaf_index'] / np.maximum(gold_train['plant_n_leaves'] - 1, 1)
gold_train['leafAngle_descriptor'] = 90.0 - gold_train['theta_gold']
binsg = np.linspace(0, 1, 12)
gold_train['_bin'] = pd.cut(gold_train['t'], binsg, include_lowest=True, labels=False)
binned_la = gold_train.groupby('_bin')['leafAngle_descriptor'].mean().reindex(range(11))
la_base = float(binned_la.iloc[5])
la_std  = float(gold_train['leafAngle_descriptor'].std())
la_bot  = float(binned_la.iloc[0])  - la_base
la_top  = float(binned_la.iloc[-1]) - la_base

# Azimuth noise: empirical median |Phi_diff - 180| from train, inverted with the
# triangular-distribution formula for uniform per-leaf noise.
_p = gold_train.sort_values(['plant_id', 'leaf_index']).copy()
_p['phi_norm'] = e1b.set_index(['plant_id','leaf_index']).loc[
    list(zip(_p['plant_id'], _p['leaf_index'])), 'phi_gold'].values % 360.0
_p['phi_next'] = _p.groupby('plant_id')['phi_norm'].shift(-1)
_diff = ((_p['phi_next'] - _p['phi_norm']) % 360.0 - 180.0).abs().dropna()
tiller_azimuth_noise = float(_diff.median() / 0.5858)  # uniform-noise inversion
print(f'\nleafAngleBase   = {la_base:.2f} +/- {la_std:.2f}  '
      f'offset bot/top = {la_bot:+.2f} / {la_top:+.2f}')
print(f'leafLengthBase  = {ll_base:.4f} +/- {ll_std:.4f}  '
      f'scale bot/top = {ll_bot:.3f} / {ll_top:.3f}')
print(f'droopinessBase  = {dr_base:.2f} +/- {dr_std:.2f}  '
      f'offset bot/top = {dr_bot:+.2f} / {dr_top:+.2f}')
print(f'tillerAzimuthNoise = {tiller_azimuth_noise:.1f}')

In [ ]:
# Build train-only config via the phenoframe.MaizeGenerator helper.
# The wrapper handles the leafLengthBase -> leafLengthScaleCurve naming
# convention correctly (the inline ET.parse path in earlier drafts
# silently no-op'd on leafLengthCurve because the actual element is
# leafLengthScaleCurve).
TRAIN_CONFIG = PAPER / 'generated' / 'sorghum_holdout_config.xml'
GENERATOR.derive_config(
    canopy_params={
        'leafLengthBase':  dict(mean=ll_base, std=ll_std,
                                bottom_scale=ll_bot,  middle_scale=1.0,   top_scale=ll_top),
        'leafAngleBase':   dict(mean=la_base, std=la_std,
                                bottom_offset=la_bot, middle_offset=0.0,  top_offset=la_top),
        'droopinessBase':  dict(mean=dr_base, std=dr_std,
                                bottom_offset=dr_bot, middle_offset=0.0,  top_offset=dr_top),
        'height':     dict(mean=height_mean,    std=height_std),
        # Sorghum-typical visual defaults (same as v3 calibration)
        'widthTaper':      dict(mean=2.2,            std=0.20),
        'leafCurl':        dict(mean=0.05,           std=0.04),
        'leafTwist':       dict(mean=0.0,            std=25.0),
        'numLeaves':       dict(mean=n_leaves_mean,  std=n_leaves_std),
    },
    base_config_path=BASE_CONFIG,
    output_config_path=TRAIN_CONFIG,
    tiller_azimuth_noise=tiller_azimuth_noise,
    leaf_jitter_scale=0.60,
)
print(f'Wrote train-only config: {TRAIN_CONFIG}')


## 3. Run the C++ generator with the held-out config

Produce 324 synthetic plants (matching the size of the original real population).

In [ ]:
SYNTH_DIR = PAPER / 'generated' / 'holdout'

t0 = time.perf_counter()
xmls = GENERATOR.generate(
    config_path=TRAIN_CONFIG,
    output_dir=SYNTH_DIR,
    count=324,
    seed_start=RNG_SEED,
    clean_first=True,
)
elapsed = time.perf_counter() - t0
print(f'C++ engine wrote {len(xmls)} plants in {elapsed:.1f}s')


## 4. Extract traits from synthetic plants via the PhenoFrame trait pipeline

Same pipeline used in §2.1 / E3 (verified equivalent to the C++ trait extractor to floating-point precision).

In [ ]:
def extract_synth_traits(xml_paths):
    rows = []
    for p in xml_paths:
        try:
            traits = compute_traits_from_descriptor(p)
        except Exception:
            continue
        for t in traits:
            az  = t['leaf_angle']['azimuth_deg']
            phi = az if az <= 180 else az - 360.0
            theta = 90.0 - t['leaf_angle']['inclination_deg']
            rows.append({
                'plant_id':    p.stem,
                'leaf_index':  t['leaf_index'],
                'theta':       theta,
                'phi':         -phi,                       # Davis sign convention
                'leaf_length': t['leaf_length'],
                'conn_y':      t['connection_point']['y'],
            })
    return pd.DataFrame(rows)

t0 = time.perf_counter()
synth_traits = extract_synth_traits(xmls)
print(f'Extracted {len(synth_traits):,} leaf-traits from {synth_traits["plant_id"].nunique()} plants '
      f'in {time.perf_counter()-t0:.1f}s')

## 5. Load the held-out test population's gold and PhenoFrame-extracted traits

Two measurements per real test plant:
- **Gold standard** (Davis voxel-PCA from `angles.txt`) — what we compare *against*.
- **PhenoFrame trait extractor applied to the E5 procedural fit** — the same trait pipeline applied to the procedurally-fit version of the SAME plant. This is the **measurement-gap baseline**: it tells us how much disagreement is built in to comparing PhenoFrame traits against voxel-PCA traits, even when the procedural parameters were directly fitted to the real skeleton.

In [ ]:
e5_roundtrip = pd.read_csv(OUT / 'e5_roundtrip_vs_gold.csv')
real_test_traits = (
    e5_roundtrip[e5_roundtrip['plant_id'].isin(test_plants)]
    [['plant_id', 'leaf_index', 'theta_gold', 'phi_gold', 'theta_rt', 'phi_rt',
      'leaf_length_rt', 'leaf_voxel_count']]
    .rename(columns={'theta_gold': 'theta_gold', 'phi_gold': 'phi_gold',
                     'theta_rt': 'theta_phenoframe', 'phi_rt': 'phi_phenoframe',
                     'leaf_length_rt': 'leaf_length_phenoframe'})
    .copy()
)
print(f'Test population: {len(real_test_traits):,} leaves, {real_test_traits["plant_id"].nunique()} plants')

## 6. Compute the comparison metrics

For each trait and each population, we compute the **plant-level aggregate** (median or mean across the lower 5 leaves) and run two distance metrics on the resulting distributions:

- **Wasserstein-1** (`W₁`, in degrees) — the average horizontal shift between the two ECDFs. Has units. Primary metric.
- **KS statistic** (sup-norm between ECDFs) — for direct comparison to E9b numbers. Secondary.

We drop KS *p-values* — at N≈260+260 any 2° mean shift is significant, so they don't differentiate "close" from "far".

Three populations compared:
- **synth**: from the C++ generator (held-out config)
- **test_gold**: real test plants, voxel-PCA gold standard
- **test_phenoframe**: real test plants, PhenoFrame trait extractor on the E5 procedural fit

Key comparisons:
- `synth ↔ test_gold` — the headline (generator vs published ground truth)
- `test_phenoframe ↔ test_gold` — the measurement-gap baseline floor
- `synth ↔ test_phenoframe` — generator vs same-pipeline-on-real (removes measurement gap)

In [ ]:
def add_phi_diff_dev(df, phi_col):
    out = df.sort_values(['plant_id','leaf_index']).copy()
    out['_phi_norm'] = out[phi_col] % 360.0
    out['_phi_next'] = out.groupby('plant_id')['_phi_norm'].shift(-1)
    diff = (out['_phi_next'] - out['_phi_norm']) % 360.0
    out['phi_diff_dev'] = np.abs(diff - 180.0)
    return out

def plant_aggs(df, theta_col, phi_col):
    aug = add_phi_diff_dev(df, phi_col)
    sub = aug[aug['leaf_index'].isin(LOWER)].copy()
    sub['_theta'] = sub[theta_col]
    return sub.groupby('plant_id').agg(
        med_theta=('_theta', 'median'),  am_theta=('_theta', 'mean'),
        med_phi_dev=('phi_diff_dev', 'median'),  am_phi_dev=('phi_diff_dev', 'mean'),
    )

agg_synth     = plant_aggs(synth_traits,    'theta',         'phi')
agg_test_gold = plant_aggs(real_test_traits, 'theta_gold',    'phi_gold')
agg_test_pym  = plant_aggs(real_test_traits, 'theta_phenoframe', 'phi_phenoframe')
print(f'Agg sizes (plants): synth={len(agg_synth)}, test_gold={len(agg_test_gold)}, test_phenoframe={len(agg_test_pym)}')

def compare(a, b):
    a = np.asarray(a)[~np.isnan(a)]; b = np.asarray(b)[~np.isnan(b)]
    if len(a) < 5 or len(b) < 5:
        return dict(W1=np.nan, KS=np.nan, mean_a=np.nan, mean_b=np.nan, std_a=np.nan, std_b=np.nan)
    return dict(
        W1=float(stats.wasserstein_distance(a, b)),
        KS=float(stats.ks_2samp(a, b).statistic),
        mean_a=float(a.mean()), mean_b=float(b.mean()),
        std_a=float(a.std(ddof=1)), std_b=float(b.std(ddof=1)),
    )

rows = []
for col in ['med_theta', 'am_theta', 'med_phi_dev', 'am_phi_dev']:
    for label, agg_target in [('synth_vs_test_gold',    agg_test_gold),
                              ('synth_vs_test_phenoframe', agg_test_pym),
                              ('measurement_gap_baseline', agg_test_pym)]:
        agg_source = agg_synth if 'synth' in label else agg_test_gold
        m = compare(agg_source[col].values, agg_target[col].values)
        rows.append({'aggregate': col, 'comparison': label, **m})
agg_compare = pd.DataFrame(rows)
pd.set_option('display.float_format', '{:.3f}'.format)
agg_compare.pivot_table(index='aggregate', columns='comparison', values=['W1', 'KS'])

### Interpreting the W₁ table

Three columns per metric:
- `synth_vs_test_gold` — **the headline**: synthetic plants vs published voxel-PCA on held-out real plants.
- `measurement_gap_baseline` — **the floor**: how much disagreement exists between PhenoFrame and voxel-PCA on the SAME held-out plants, with no generation involved.
- `synth_vs_test_phenoframe` — synthetic plants vs same-pipeline real (removes measurement-method gap).

**Reading rule.** If `synth_vs_test_gold` ≈ `measurement_gap_baseline`, the generator is at the floor — its residual is fully explained by the published-pipeline-vs-PhenoFrame disagreement, not by population mismatch. If `synth_vs_test_phenoframe` is small while the gold comparison is large, the generator matches the *test* population well; the apparent error is the measurement-method gap from §2.2.

## 7. Visualization: ECDF panels

Each panel shows three ECDFs on the same axes for one plant-aggregate. Wasserstein-1 in the title gives the average horizontal gap in degrees.

In [ ]:
def ecdf(x):
    xs = np.sort(np.asarray(x)[~np.isnan(x)])
    ys = np.arange(1, len(xs) + 1) / len(xs)
    return xs, ys

labels_pretty = {
    'med_theta':   r'plant median $\theta$ (deg)',
    'am_theta':    r'plant mean $\theta$ (deg)',
    'med_phi_dev': r'plant median $|\Delta\phi - 180^\circ|$ (deg)',
    'am_phi_dev':  r'plant mean $|\Delta\phi - 180^\circ|$ (deg)',
}

fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
for ax, col in zip(axes, ['med_theta', 'am_theta', 'med_phi_dev', 'am_phi_dev']):
    for label, agg, color, ls in [
        ('real test (gold)',      agg_test_gold, '#222222', '-'),
        ('real test (PhenoFrame)',   agg_test_pym,  '#888888', '--'),
        ('synthetic (C++)',       agg_synth,     '#cc6677', '-'),
    ]:
        xs, ys = ecdf(agg[col].dropna().values)
        ax.plot(xs, ys, lw=2.0, color=color, linestyle=ls, label=label)
    # Pull W1 of headline comparison from the table
    w1 = agg_compare.query(f"aggregate == '{col}' and comparison == 'synth_vs_test_gold'").iloc[0]['W1']
    floor = agg_compare.query(f"aggregate == '{col}' and comparison == 'measurement_gap_baseline'").iloc[0]['W1']
    ax.set_xlabel(labels_pretty[col])
    ax.set_ylabel('ECDF')
    ax.set_title(f'$W_1$(synth, gold) = {w1:.2f}° vs floor {floor:.2f}°', fontsize=10)
    ax.grid(alpha=0.3)
    if ax is axes[0]:
        ax.legend(loc='lower right', fontsize=9)
fig.suptitle('Held-out validation: ECDFs of plant-level aggregates  '
             '(real test set vs synthetic)', y=1.02, fontsize=12)
fig.tight_layout()
fig.savefig(FIG / 'sec2_5_holdout_aggregate_ecdfs.png', dpi=140, bbox_inches='tight')
plt.show()

## 8. Per-node distributions with W₁

Wasserstein-1 distance for θ and phyllotaxic deviation at each lower-5 node, across the three population pairs. Lets us see where the generator does best and worst along the canopy.

In [ ]:
synth_aug = add_phi_diff_dev(synth_traits.rename(columns={'phi': 'phi'}), 'phi')
test_gold_aug = add_phi_diff_dev(real_test_traits.rename(columns={'phi_gold': 'phi'}),    'phi')
test_pym_aug  = add_phi_diff_dev(real_test_traits.rename(columns={'phi_phenoframe': 'phi'}), 'phi')
test_gold_aug['theta'] = real_test_traits['theta_gold'].values
test_pym_aug['theta']  = real_test_traits['theta_phenoframe'].values

rows = []
for trait in ['theta', 'phi_diff_dev']:
    for node in LOWER:
        s = synth_aug.loc[synth_aug['leaf_index']==node, trait].dropna().values
        g = test_gold_aug.loc[test_gold_aug['leaf_index']==node, trait].dropna().values
        p = test_pym_aug.loc[test_pym_aug['leaf_index']==node, trait].dropna().values
        rows.append({
            'trait': trait, 'node': node,
            'W1_synth_vs_gold':    compare(s, g)['W1'],
            'W1_synth_vs_phenoframe': compare(s, p)['W1'],
            'W1_floor':            compare(g, p)['W1'],
        })
per_node_w1 = pd.DataFrame(rows)
per_node_w1

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharex=True)
for ax, trait, title in zip(axes,
                            ['theta', 'phi_diff_dev'],
                            [r'$\theta$ (deg)', r'$|\Delta\phi - 180^\circ|$ (deg)']):
    sub = per_node_w1[per_node_w1['trait']==trait]
    nodes = sub['node'].values
    ax.plot(nodes, sub['W1_synth_vs_gold'],    '-o', color='#cc6677', label='synth vs gold (headline)')
    ax.plot(nodes, sub['W1_synth_vs_phenoframe'], '-s', color='#4477aa', label='synth vs PhenoFrame-on-real')
    ax.plot(nodes, sub['W1_floor'],            '--', color='#222222', label='measurement gap floor')
    ax.set_xlabel('leaf node')
    ax.set_ylabel('$W_1$ (deg)')
    ax.set_title(title)
    ax.grid(alpha=0.3)
    if ax is axes[0]:
        ax.legend(loc='best', fontsize=9)
fig.suptitle('Per-node $W_1$ distance, generator vs gold vs measurement-gap floor', y=1.02, fontsize=12)
fig.tight_layout()
fig.savefig(FIG / 'sec2_5_holdout_per_node_w1.png', dpi=140, bbox_inches='tight')
plt.show()

## 9. Save outputs

In [ ]:
agg_compare.to_csv(OUT / 'sec2_5_holdout_aggregate_compare.csv', index=False)
per_node_w1.to_csv(OUT / 'sec2_5_holdout_per_node_w1.csv',       index=False)
synth_traits.to_csv(OUT / 'sec2_5_holdout_synthetic_traits.csv', index=False)
pd.DataFrame({'plant_id': sorted(train_plants), 'split': 'train'}).to_csv(
    OUT / 'sec2_5_holdout_split.csv', index=False)
pd.concat([
    pd.DataFrame({'plant_id': sorted(train_plants), 'split': 'train'}),
    pd.DataFrame({'plant_id': sorted(test_plants),  'split': 'test'}),
]).to_csv(OUT / 'sec2_5_holdout_split.csv', index=False)

print('Wrote outputs:')
for f in ['sec2_5_holdout_aggregate_compare.csv','sec2_5_holdout_per_node_w1.csv',
          'sec2_5_holdout_synthetic_traits.csv','sec2_5_holdout_split.csv']:
    print(f'  {OUT / f}')
print(f'  config used: {TRAIN_CONFIG}')
print(f'  synthetic XML/OBJ: {SYNTH_DIR}')
print('Wrote figures:')
for f in ['sec2_5_holdout_aggregate_ecdfs.png','sec2_5_holdout_per_node_w1.png']:
    print(f'  {FIG / f}')

## Discussion

**What this validates.** The C++ generator, **calibrated on a train-only 80% split** and never shown the test plants, produces synthetic plants whose plant-level aggregate trait distributions are compared against:
- the held-out test plants' published voxel-PCA gold standard (headline), and
- the measurement-gap baseline (PhenoFrame trait pipeline applied to the procedural fit of the SAME held-out plants — no generation involved).

**Reading the headline numbers.** For each aggregate the $W_1$ column shows:
- if `synth_vs_test_gold` $\approx$ `measurement_gap_baseline`, the generator's residual is dominated by the published-vs-PhenoFrame measurement gap (the gap documented in §2.2 / E1b at R²(θ)=0.48, R²(φ)=0.53). The generator is at the measurement-method floor and cannot be improved without changing the trait extractor.
- if `synth_vs_test_gold` is much larger than the floor, the residual is genuine generator-vs-population mismatch.
- if `synth_vs_test_phenoframe` is small even while `synth_vs_test_gold` is large, the generator matches the *same-pipeline real* distribution well, and the gold-comparison gap is entirely measurement-method.

**Why $W_1$ instead of KS p-value.** At N $\approx$ 260 vs 65, KS p-values reject the null for any 2° mean shift; they don't differentiate "close to the floor" from "far above it". $W_1$ has units (degrees) and reports the *magnitude* of disagreement.

**Calibration honesty.** The three knobs tuned in `08_sec2.5_data_driven_generation.ipynb` (`leafAngleBase`, `tillerAzimuthNoise`, `leafJitterScale`) are not re-tuned here. The first two are deterministic functions of the train-set statistics; only `leafJitterScale = 0.60` is a one-time choice that was justified in `08_*` and held fixed. The other 10+ generator parameters (loadings, whorl, visual defaults) are pulled from the maize config without alteration.

**What this experiment does not address.**
- The 4-latent (V/S/P/T) structure is still inherited from the maize config without independent validation in sorghum. A null-model comparison (independent per-parameter Gaussians) would test whether the latent factor architecture is necessary.
- The held-out trait set is θ and φ only. We do not compare leaf_length, internode distance, or plant height because the gold-standard `angles.txt` does not include them, so a clean validation target does not exist.
- The 80/20 split is one random partition. A k-fold version would tighten confidence intervals on the per-aggregate $W_1$ estimates.

**Outputs.** `sec2_5_holdout_aggregate_compare.csv` (W₁ + KS at the plant-aggregate level), `sec2_5_holdout_per_node_w1.csv` (per-node W₁ across the three comparisons), `sec2_5_holdout_synthetic_traits.csv`, `sec2_5_holdout_split.csv`. Figures: `sec2_5_holdout_aggregate_ecdfs.png`, `sec2_5_holdout_per_node_w1.png`.